In [ ]:
# ============================================================
# INSTALAÇÃO DE BIBLIOTECAS
# ============================================================
import cv2
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial

#criação da classe
class pipeline_Processor:
    """
    Pipeline de pré-processamento para peças de fundição (Casting Dataset).
    Grayscale -> Blur -> Threshold (Otsu) -> Canny -> Morfologia -> Resize
    """

    def __init__(self, blur_kernel=(5, 5), canny_thresh1=50, canny_thresh2=150,
                 morph_kernel_size=(3, 3), output_size=(256, 256)):
        self.blur_kernel = blur_kernel
        self.canny_thresh1 = canny_thresh1
        self.canny_thresh2 = canny_thresh2
        self.morph_kernel_size = morph_kernel_size
        self.output_size = output_size
        self.valid_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tiff", ".tif")

    # 1. Mostrar imagem (debug)
    def _show_image(self, img, file_name="Image"):
        try:
            cv2.imshow(file_name, img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except Exception as e:
            print(f"[Erro ao exibir imagem]: {e}")

    # 2. Leitura
    def _read_image(self, path):
        try:
            img = cv2.imread(path)
            if img is None:
                print(f"[Aviso] Imagem não carregada ou corrompida: {path}")
            return img
        except Exception as e:
            print(f"[Erro ao ler imagem]: {e}")
            return None

    # 3. Grayscale (Sprint 3)
    def _to_gray_scale(self, img):
        try:
            return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        except Exception as e:
            print(f"[Erro na conversão para escala de cinza]: {e}")
            return None

    # 4. Blur (Sprint 3)
    def _apply_blur(self, img):
        try:
            return cv2.GaussianBlur(img, self.blur_kernel, 0)
        except Exception as e:
            print(f"[Erro na suavização gaussiana]: {e}")
            return None

    # 5. Threshold Otsu (Sprint 4)
    def _apply_threshold_otsu(self, img):
        try:
            _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            return binary
        except Exception as e:
            print(f"[Erro na limiarização]: {e}")
            return None

    # 6. Canny (Sprint 4)
    def _apply_canny(self, img):
        try:
            return cv2.Canny(img, self.canny_thresh1, self.canny_thresh2)
        except Exception as e:
            print(f"[Erro na detecção de bordas]: {e}")
            return None

    # 7. Morfologia (Sprint 5)
    def _apply_morphology(self, img):
        try:
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, self.morph_kernel_size)
            opened = cv2.morphologyEx(img, cv2.MORPH_OPEN, kernel)
            closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
            return closed
        except Exception as e:
            print(f"[Erro na etapa de operações morfológicas]: {e}")
            return None

    # 8. Resize (Sprint 5)
    def _resize_image(self, img):
        try:
            return cv2.resize(img, self.output_size, interpolation=cv2.INTER_AREA)
        except Exception as e:
            print(f"[Erro no redimensionamento padrão]: {e}")
            return None

    # 9. Salvar (Sprint 6)
    def _save_image(self, img, path):
        try:
            cv2.imwrite(path, img)
            return True
        except Exception as e:
            print(f"[Erro ao salvar imagem]: {e}")
            return False

    # ============================================================
    # ORQUESTRAÇÃO DE UMA IMAGEM
    # ============================================================
    def process_single_image(self, input_path, output_path, mostrar=False):
    # 1. Leitura
        img = self._read_image(input_path)
        if img is None:
            return False

        # 2. Grayscale
        gray = self._to_gray_scale(img)
        if gray is None:
            return False

        # 3. Blur
        blurred = self._apply_blur(gray)
        if blurred is None:
            return False

        # 4. Threshold com Otsu → máscara binária
        binary = self._apply_threshold_otsu(blurred)
        if binary is None:
            return False

        # 5. Morfologia sobre a máscara (refina a segmentação)
        mask_clean = self._apply_morphology(binary)
        if mask_clean is None:
            return False

        # 6. Aplica a máscara na imagem blurred
        #    → só a região da peça permanece; o fundo fica preto
        masked = cv2.bitwise_and(blurred, blurred, mask=mask_clean)

        # 7. Canny sobre a imagem MASCARADA (mostra ranhuras internas sem ruído do fundo)
        edges = self._apply_canny(masked)
        if edges is None:
            return False

        # 8. Resize
        final = self._resize_image(edges)
        if final is None:
            return False

        if mostrar:
            self._show_image(img, "1 - Original")
            self._show_image(gray, "2 - Grayscale")
            self._show_image(blurred, "3 - Gaussian Blur")
            self._show_image(binary, "4 - Threshold Otsu")
            self._show_image(mask_clean, "5 - Morfologia (máscara)")
            self._show_image(masked, "6 - Imagem mascarada")
            self._show_image(edges, "7 - Bordas Canny")
            self._show_image(final, "8 - Resultado Final")

        return self._save_image(final, output_path)
    # ============================================================
    # WRAPPER (ThreadPool não precisa de staticmethod)
    # ============================================================
    def _process_wrapper(self, filename, input_dir, output_dir):
        try:
            in_path = os.path.join(input_dir, filename)
            base_name = os.path.splitext(filename)[0]
            out_path = os.path.join(output_dir, f"{base_name}.png")
            return self.process_single_image(in_path, out_path)
        except Exception as e:
            import traceback
            print(f"[Erro ao processar {filename}]: {e}")
            traceback.print_exc()
            return False

    # ============================================================
    # PROCESSAMENTO EM LOTE — THREADS
    # ============================================================
    def process_batch_concurrent(self, input_dir, output_dir, limit=None):
        try:
            Path(output_dir).mkdir(parents=True, exist_ok=True)

            files = [
                f for f in os.listdir(input_dir)
                if f.lower().endswith(self.valid_extensions)
            ]
            if limit is not None:
                files = files[:limit]

            if not files:
                print(f"Não há imagens válidas em {input_dir}")
                return

            max_workers = max(1, (os.cpu_count() or 2) - 1)
            print(f"Iniciando pré-processamento de {len(files)} imagens com {max_workers} threads...")

            sucesso = 0
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                func = partial(
                    self._process_wrapper,
                    input_dir=input_dir,
                    output_dir=output_dir
                )
                futures = [executor.submit(func, filename) for filename in files]

                for i, future in enumerate(as_completed(futures), 1):
                    try:
                        if future.result(timeout=60):
                            sucesso += 1
                    except Exception as e:
                        print(f"\n[Erro em thread]: {e}")
                    print(f"\r[{i}/{len(files)}] processadas", end="")

            print(f"\n{sucesso}/{len(files)} imagens processadas com sucesso")

        except Exception as e:
            print(f"Erro ao processar em lote: {e}")




In [ ]:
#instanciando a classe: processo em lote

pipeline = pipeline_Processor()

pipeline.process_batch_concurrent(
        "data/raw_images/def_front",
        "output/processed_images/def_front",)

pipeline.process_batch_concurrent(
        "data/raw_images/ok_front",
        "output/processed_images/ok_front",)

Iniciando pré-processamento de 781 imagens com 3 threads...
[781/781] processadas
781/781 imagens processadas com sucesso
Iniciando pré-processamento de 519 imagens com 3 threads...
[519/519] processadas
519/519 imagens processadas com sucesso


In [ ]:
#Instanciando a classe para chamar e mostrar/salvar uma unica imagem

#pipeline=pipeline_Processor()
# pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_k7.jpeg')
# pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_k5.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_canny_120_240.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_canny_30_100.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_1021.jpeg','output/processed_images/test/cast_ok_0_1021_canny_50_150.jpeg')
#pipeline.process_single_image('data/raw_images/def_front/cast_def_0_2234.jpeg','output/processed_images/test/cast_def_0_2234_canny_120_240.jpeg')
#pipeline.process_single_image('data/raw_images/def_front/cast_def_0_2234.jpeg','output/processed_images/test/cast_def_0_2234_test.jpeg')


#testando pipeline imagem unica
# processador = pipeline_Processor()
# caminho_entrada = "data/raw_images/def_front/cast_def_0_9477.jpeg"
# caminho_saida = "data/processed_images/test/cast_def_0_9477_processada.jpeg"
# processador.process_single_image(
#      caminho_entrada,
#      caminho_saida,
#     mostrar=True
# )
#image=pipeline._read_image('data/raw_images/def_front/cast_def_0_146.jpeg')


#testando images
# img_cinza=pipeline._to_gray_scale(image)
# img_blur=pipeline._apply_blur(img_cinza)
# img_thresh=pipeline._apply_threshold_otsu(img_blur)
# img_bordas=pipeline._apply_canny(img_thresh)
# img_morfologia=pipeline._apply_morphology(img_bordas)
# img_resize=pipeline._resize_image(img_morfologia)

# pipeline._save_image(img_resize,'output/processed_iamges/test/cast_def_0_146.jpeg')
#pipeline._show_image(img_resize,'cast_def_0_146.jpeg_resize')
#pipeline._show_image(image,'cast_def_0_146.jpeg_original')0
# pipeline._read_image('data/raw_images/def_front/cast_def_0_146.jpeg')


#pipeline unico

True